# OCSD – Chạy toàn bộ thực nghiệm (Run all)

Khóa luận: *Sinh ảnh cảnh nhất quán đối tượng từ phác thảo và văn bản bằng mô hình khuếch tán* – Lê Hoàng Bảo (21090331), Thái Bá Cường (21050681); GVHD: TS. Nguyễn Thanh Chuyên.

**Cách dùng:** Runtime → Change runtime type → GPU (L4 hoặc A100) → sửa ô 1 → Runtime → Run all.

Notebook chạy 4 giai đoạn: (A) dựng bộ dữ liệu, (B) sinh ảnh cho E1–E4 và khảo sát α, (C) đánh giá, (D) xuất bảng/hình/tóm tắt. Mọi kết quả nằm trong `DRIVE_ROOT/results` (`summary.md`, `summary.json`, `tables/`, `figures/`, `progress.json`). Nếu Colab ngắt kết nối, chỉ cần Run all lại: phần đã xong được bỏ qua.

In [ ]:
#@title 1. Cấu hình (chỉ sửa ô này)
REPO_URL   = "https://github.com/bobbibao/ocsd-thesis.git"  #@param {type:"string"}
BRANCH     = "main"                                             #@param {type:"string"}
DRIVE_ROOT = "/content/drive/MyDrive/KLTN_OCSD"                 #@param {type:"string"}
TIER       = "pilot"   #@param ["pilot", "paper", "full"]
BACKBONE   = "sd15"    #@param ["sd15", "sd21"]
MAX_MINUTES = 600      #@param {type:"number"}
HF_TOKEN   = ""        #@param {type:"string"}
# pilot: kiểm tra nhanh (~1-2 giờ GPU); paper: đủ số liệu cho khóa luận; full: cho bài báo (cần thêm GPU).
# Có thể bấm "Run all" nhiều lần: ảnh/độ đo đã có trên Drive sẽ được bỏ qua và chạy tiếp phần còn lại.

In [ ]:
#@title 2. Gắn Google Drive, kiểm tra GPU, lấy mã nguồn mới nhất
import os, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_ROOT, exist_ok=True)
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv", shell=True, capture_output=True, text=True).stdout)
CODE = "/content/KLTN_OCSD"
if not REPO_URL:
    # chưa có repo: dùng bản mã nguồn đã tải lên Drive (thư mục DRIVE_ROOT/code)
    CODE = os.path.join(DRIVE_ROOT, "code")
elif os.path.exists(CODE):
    subprocess.run(f"git -C {CODE} fetch -q origin {BRANCH} && git -C {CODE} reset -q --hard origin/{BRANCH}", shell=True, check=True)
else:
    subprocess.run(f"git clone -q -b {BRANCH} {REPO_URL} {CODE}", shell=True, check=True)
print("Mã nguồn:", CODE, subprocess.run(f"git -C {CODE} log -1 --format='%h %s' 2>/dev/null", shell=True, capture_output=True, text=True).stdout)
sys.path.insert(0, CODE)
# drop modules imported by an earlier run in this kernel, so the code just pulled is what runs
for _m in [m for m in sys.modules if m == "ocsd" or m.startswith("ocsd.")]: del sys.modules[_m]
# HF cache on LOCAL disk: Drive cannot store the cache's symlinks (broken files -> "SafetensorError: header too large").
# Models re-download each new session (~2-4 min on Colab).
os.environ["HF_HOME"] = "/content/hf_cache"
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

In [ ]:
#@title 3. Install libraries (keeps Colab's PyTorch), start the Drive run log
LOGS = os.path.join(DRIVE_ROOT, "results", "logs"); os.makedirs(LOGS, exist_ok=True)
# Colab ships torchao 0.10, which peft rejects when adding LoRA ("incompatible version of torchao"); OCSD does not use it
subprocess.run("pip uninstall -y -q torchao", shell=True)
r = subprocess.run(f"pip install -q -r {CODE}/requirements-colab.txt", shell=True, capture_output=True, text=True)
open(os.path.join(LOGS, "pip_install.log"), "w").write(r.stdout + "\n" + r.stderr)
print(r.stdout[-2000:], r.stderr[-3000:])
if r.returncode != 0:
    raise RuntimeError(f"pip install failed; see {LOGS}/pip_install.log")
import importlib, ocsd; importlib.reload(ocsd)
from ocsd import hfcache
hfcache.repair(os.environ["HF_HOME"])   # delete broken model files so they download again
from ocsd import runlog
runlog.start(os.path.join(DRIVE_ROOT, "results"), CODE)   # every print below also goes to results/logs/run_*.log
from ocsd.config import Paths, ExperimentConfig, TIERS
P = Paths(DRIVE_ROOT); P.makedirs()
E = ExperimentConfig(backbone=BACKBONE, tier=TIER)
E.save(os.path.join(P.results, "experiment_config.json"))
print(E)

## A. Bộ dữ liệu
QuickDraw-Scenes (tổng hợp, 12 ô: số đối tượng 1/3/5/8+ × độ phức tạp đơn giản/trung bình/phức tạp) và COCO-Sketch (ảnh thật COCO val2017, phác thảo PiDiNet, mặt nạ thật). Lần đầu mất ~10–20 phút (tải COCO 1 GB).

In [ ]:
#@title A. Build benchmarks (skipped if already built for this tier)
from ocsd import stages
with runlog.stage("A_data"):
    stages.setup_data(P, TIER, with_coco=True)
from IPython.display import Image, display
f = os.path.join(P.results, "figures", "benchmark_examples.png")
if os.path.exists(f): display(Image(f, width=900))

## B. Sinh ảnh
Thứ tự: E3 QuickDraw (mọi baseline + OCSD-lite trên toàn bộ tập) → OCSD và Zhang et al. (cần học định danh, trên tập con phân tầng) → E3 COCO → E4 cắt bỏ → khảo sát α. `MAX_MINUTES` giới hạn thời gian của ô này; chạy lại để tiếp tục.

## T. Tuning
Paper/full tiers only: choose α and the LoRA strength on the pilot scenes, which are then left out of the evaluation. Runs once (about 1 A100 hour) and writes `results/tuning/tuned.json`; skipped afterwards.

In [ ]:
#@title T. Tune alpha and LoRA strength on the pilot scenes (skipped when done)
import torch
eng = vis = None
with runlog.stage("T_tune"):
    eng, vis = stages.tune(P, E, max_minutes=MAX_MINUTES)

In [ ]:
#@title B1. Plan and remaining images
for j in stages.experiment_plan(P, E):
    print(f"{j['name']:22s} {len(j['scenes']):4d} scenes x {len(j['methods']):2d} methods x {len(j['seeds'])} seeds -> {stages._count_todo(P, j)} images left")

In [ ]:
#@title B2. Generate images (resumable: finished images are skipped)
with runlog.stage("B_generate"):
    eng, vis = stages.generate(P, E, max_minutes=MAX_MINUTES, eng=eng, vis=vis)

## C. Đánh giá
OPR, OCE, độ chính xác đếm, mIoU, RA bằng bộ phát hiện **OWLv2** (khác Grounding DINO mà phương pháp dùng bên trong, để tránh tối ưu hóa theo chính độ đo); CLIP score toàn ảnh và cấp đối tượng; ID-Sim (DINOv2); FID/KID; LPIPS.

In [ ]:
#@title C. Metrics (only new images are evaluated; FID reused when unchanged)
if eng is not None:
    del eng; torch.cuda.empty_cache()
with runlog.stage("C_evaluate"):
    vis = stages.evaluate(P, E, vis=vis, fid=True)

## D. Báo cáo

In [ ]:
#@title D. Tables, figures, statistics, summary
with runlog.stage("D_report"):
    summary = stages.report(P, E)
    from IPython.display import Markdown, display
    display(Markdown(open(os.path.join(P.results, "summary.md")).read()))
    for f in summary["figures"]:
        display(Image(f, width=900))

In [ ]:
#@title E. Run status (also saved to results/logs/status.json; errors in results/logs/errors/)
runlog.summary()
if os.path.exists(os.path.join(LOGS, "LATEST_ERROR.txt")):
    print("\nMost recent error:\n" + open(os.path.join(LOGS, "LATEST_ERROR.txt")).read()[-3000:])

In [ ]:
#@title F. Send logs and results to GitHub (branch colab-results) so Claude can read them
from ocsd import sync
try:
    sync.push_results(P.results, sync.colab_token())
except Exception as e:
    print("[sync] failed:", e)